# Fine-tuning QLoRA de MedGemma-4b sur RSNA (Kaggle)

> **Prototype pédagogique. Non destiné au diagnostic.**

Ce notebook adapte **`google/medgemma-4b-it`** par **QLoRA** (4-bit + LoRA) pour
qu'il produise, sur une radiographie thoracique frontale, le JSON du projet avec
la bonne classe (`normal` / `suspected_opacity`). On entraîne l'adaptateur, on
l'évalue vs le zero-shot, puis on télécharge l'adaptateur pour l'utiliser en local.

## Prérequis Kaggle (à faire dans le panneau de droite)
1. **Settings → Accelerator → GPU T4 x1** (ou P100).
2. **Settings → Internet → On**.
3. **Add Input → Competitions →** *RSNA Pneumonia Detection Challenge*
   (accepte d'abord les règles de la compétition sur sa page).
4. MedGemma est **sous accès contrôlé** : accepte la licence sur
   https://huggingface.co/google/medgemma-4b-it, crée un **token HF**, puis
   **Add-ons → Secrets** → ajoute un secret `HF_TOKEN` avec ce token.

Temps indicatif : ~20-40 min sur T4 pour ~300 exemples, 1-2 epochs.

In [ ]:
# 0. Réduire la fragmentation mémoire GPU (avant l'import de torch)
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# 1. Dépendances (Kaggle a déjà torch+CUDA)
!pip -q install -U "transformers>=4.50" "trl>=0.12" "peft>=0.13" "bitsandbytes>=0.43" accelerate datasets pydicom

In [ ]:
# 2. Accès Hugging Face (MedGemma est gated)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
# 3. Config (auto-détection du dossier RSNA monté par Kaggle)
import os, glob
print("Contenu de /kaggle/input :", os.listdir("/kaggle/input"))

hits = glob.glob("/kaggle/input/**/stage_2_detailed_class_info.csv", recursive=True)
assert hits, ("Données RSNA introuvables. Panneau droit -> 'Add Input' -> compétition "
              "'RSNA Pneumonia Detection Challenge' (accepte d'abord ses règles).")
CLASS_CSV = hits[0]
RSNA = os.path.dirname(CLASS_CSV)
TRAIN_IMG_DIR = os.path.join(RSNA, "stage_2_train_images")

MODEL_ID = "google/medgemma-4b-it"
PER_CLASS = 200        # exemples par classe (normal / opacity)
VAL_FRAC  = 0.15
OUTPUT_DIR = "/kaggle/working/medgemma-rsna-lora"
print("RSNA:", RSNA)
print("dossier images présent :", os.path.isdir(TRAIN_IMG_DIR))

In [ ]:
# 4. Cibles JSON (identiques à finetuning/build_training_data.py du dépôt)
import json
WARNING = "Prototype pédagogique. Non destiné au diagnostic. Validation par un professionnel qualifié requise."
SYSTEM_PROMPT = ("You are an educational radiology assistant for engineering students. "
    "You are not a clinician and must never provide a definitive diagnosis. "
    "You only describe what is visible on the provided frontal chest X-ray and "
    "you always answer with a single valid JSON object, nothing else.")
USER_PROMPT = ("Analyze the provided frontal chest X-ray: normal vs suspected lung opacity vs uncertain. "
    "Return only valid JSON with keys image_quality, predicted_class, confidence, "
    "visual_evidence, justification, limitations, warning.")

TEMPLATES = {
 "normal": [
   {"image_quality":"good","confidence":0.82,
    "visual_evidence":["clear lung fields","no focal opacity or consolidation"],
    "justification":"The lung fields appear clear without a focal opacity or consolidation on this frontal view. No definitive abnormality is visible."},
   {"image_quality":"good","confidence":0.78,
    "visual_evidence":["symmetric lucency of both lungs","no visible airspace opacity"],
    "justification":"Both lungs show symmetric lucency and no visible airspace opacity. The appearance is within normal limits on this single projection."},
 ],
 "suspected_opacity": [
   {"image_quality":"good","confidence":0.76,
    "visual_evidence":["area of increased opacity in a lung field","loss of normal lucency"],
    "justification":"There is an area of increased opacity with loss of normal lucency in a lung field, which could correspond to airspace disease. This is an educational observation, not a diagnosis."},
   {"image_quality":"good","confidence":0.71,
    "visual_evidence":["hazy opacity projected over a lung zone","ill-defined margins"],
    "justification":"A hazy, ill-defined opacity is projected over a lung zone. Such an appearance can be seen with a pulmonary opacity, but projection and technique should be considered."},
 ],
}

def target_json(label, idx):
    v = TEMPLATES[label][idx % len(TEMPLATES[label])]
    return json.dumps({
        "image_quality": v["image_quality"], "predicted_class": label,
        "confidence": v["confidence"], "visual_evidence": v["visual_evidence"],
        "justification": v["justification"],
        "limitations": ["single frontal view","no clinical context","not a validated medical model"],
        "warning": WARNING}, ensure_ascii=False)

In [ ]:
# 5. Construire le dataset (DICOM -> image PIL, split train/val équilibré)
import csv, random
import numpy as np, pydicom
from PIL import Image
from datasets import Dataset

CLASS_MAP = {"Normal":"normal","Lung Opacity":"suspected_opacity"}

def load_dicom(pid, size=512):
    ds = pydicom.dcmread(os.path.join(TRAIN_IMG_DIR, pid + ".dcm"))
    a = ds.pixel_array.astype("float32")
    lo, hi = a.min(), a.max()
    if hi > lo: a = (a - lo) / (hi - lo) * 255.0
    return Image.fromarray(a.astype("uint8")).convert("RGB").resize((size, size))

seen=set(); buckets={"normal":[],"suspected_opacity":[]}
for r in csv.DictReader(open(CLASS_CSV)):
    pid=r["patientId"]
    if pid in seen: continue
    seen.add(pid); lab=CLASS_MAP.get(r["class"])
    if lab: buckets[lab].append(pid)

rng=random.Random(7); picked=[]
for lab,pids in buckets.items():
    rng.shuffle(pids); picked += [(p,lab) for p in pids[:PER_CLASS]]
rng.shuffle(picked)

records=[]
for i,(pid,lab) in enumerate(picked):
    records.append({"image": load_dicom(pid), "label": lab, "target": target_json(lab, i)})
n_val=int(len(records)*VAL_FRAC)
val_ds=Dataset.from_list(records[:n_val]); train_ds=Dataset.from_list(records[n_val:])
print("train:", len(train_ds), "val:", len(val_ds))

In [ ]:
# 6. Charger MedGemma en 4-bit (QLoRA) — forcer UN seul GPU
# device_map={"": 0} évite le sharding sur T4 x2 (sinon RuntimeError cross-device
# dans la loss). Le modèle 4B en 4-bit tient sur un seul T4 (16 Go).
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, quantization_config=bnb, torch_dtype=torch.bfloat16, device_map={"": 0})

# Étape QLoRA standard : active le gradient checkpointing correctement et
# prépare les couches k-bit -> réduit fortement la mémoire d'activation.
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False})

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "right"

In [ ]:
# 7. LoRA (langage seulement ; tour de vision gelée) + collator qui masque le prompt
from peft import LoraConfig

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

IMAGE_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids(
    getattr(processor.tokenizer, "image_token", "<image_soft_token>"))

def build_messages(example, with_answer):
    content=[{"type":"text","text":USER_PROMPT},{"type":"image","image":example["image"]}]
    msgs=[{"role":"system","content":[{"type":"text","text":SYSTEM_PROMPT}]},
          {"role":"user","content":content}]
    if with_answer:
        msgs.append({"role":"assistant","content":[{"type":"text","text":example["target"]}]})
    return msgs

def collate_fn(examples):
    input_ids_list, labels_list, pixel_list = [], [], []
    for ex in examples:
        full = processor.apply_chat_template(build_messages(ex, True),
                 add_generation_prompt=False, tokenize=True, return_dict=True, return_tensors="pt")
        prompt = processor.apply_chat_template(build_messages(ex, False),
                 add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
        ids = full["input_ids"][0]
        labels = ids.clone()
        plen = prompt["input_ids"].shape[-1]
        labels[:plen] = -100                       # ne pas apprendre le prompt
        labels[ids == processor.tokenizer.pad_token_id] = -100
        labels[ids == IMAGE_TOKEN_ID] = -100        # ni les tokens image
        input_ids_list.append(ids); labels_list.append(labels)
        pixel_list.append(full["pixel_values"][0])
    pad = processor.tokenizer.pad_token_id
    maxlen = max(x.shape[-1] for x in input_ids_list)
    def padseq(x, fill):
        import torch
        return torch.cat([x, torch.full((maxlen-x.shape[-1],), fill, dtype=x.dtype)])
    batch = {
      "input_ids": torch.stack([padseq(x, pad) for x in input_ids_list]),
      "labels":    torch.stack([padseq(x, -100) for x in labels_list]),
      "attention_mask": torch.stack([padseq(torch.ones_like(x), 0) for x in input_ids_list]),
      "pixel_values": torch.stack(pixel_list),
    }
    return batch

In [ ]:
# 8. Entraînement (QLoRA, bf16 : évite le GradScaler fp16 qui casse sur grads bf16)
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir=OUTPUT_DIR, num_train_epochs=2,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=2e-4, warmup_ratio=0.03, lr_scheduler_type="cosine",
    logging_steps=10, save_strategy="epoch", fp16=False, bf16=True,
    gradient_checkpointing=False,  # déjà activé via prepare_model_for_kbit_training
    optim="paged_adamw_8bit", report_to="none", dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds,
                     peft_config=peft_config, data_collator=collate_fn)
trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("adapter saved to", OUTPUT_DIR)

## 9. Évaluation rapide (adaptateur vs zero-shot) sur le split validation

On génère le JSON pour quelques cas de validation avec le modèle **fine-tuné** et
on compare la classe prédite à la vérité terrain. Pour une évaluation complète,
téléchargez l'adaptateur et rejouez `eval/run_evaluation.py` en local (voir
`finetuning/README.md`).

In [ ]:
# 9. Sanity-check sur le split val
import re

# Repasser en mode inférence : sans ça, le gradient checkpointing force use_cache=False
# et la génération multimodale casse ("Tensors must have same number of dimensions").
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

def predict_class(example):
    msgs = build_messages(example, with_answer=False)
    inp = processor.apply_chat_template(msgs, add_generation_prompt=True,
            tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
    n = inp["input_ids"].shape[-1]
    out = model.generate(**inp, max_new_tokens=200, do_sample=False)
    txt = processor.decode(out[0][n:], skip_special_tokens=True)
    m = re.search(r'"predicted_class"\s*:\s*"([a-z_]+)"', txt)
    return m.group(1) if m else "parse_error"

ok=0; N=min(20, len(val_ds))
for i in range(N):
    ex=val_ds[i]; pred=predict_class(ex); ok += (pred==ex["label"])
    print(f"{i:02d} vrai={ex['label']:<18} prédit={pred}")
print(f"\naccuracy fine-tuné (val, {N} cas): {ok/N:.2f}")

In [ ]:
# 10. Empaqueter l'adaptateur pour téléchargement local
import shutil
shutil.make_archive("/kaggle/working/medgemma_rsna_lora", "zip", OUTPUT_DIR)
print("Télécharge /kaggle/working/medgemma_rsna_lora.zip depuis l'onglet Output.")

## Ensuite (en local)
1. Télécharge `medgemma_rsna_lora.zip`, décompresse-le dans le dépôt sous
   `finetuning/adapter/`.
2. Lance l'app / l'éval avec l'adaptateur :
   ```bash
   set MEDGEMMA_ADAPTER=finetuning/adapter        # PowerShell: $env:MEDGEMMA_ADAPTER="finetuning/adapter"
   python eval/run_evaluation.py --engine medgemma --mode baseline --cases data/rsna_cases.csv --out-dir eval/results_ft
   ```
3. Compare `eval/results_ft` au zero-shot pour chiffrer le gain du fine-tuning.

> Le système reste **non clinique** : le fine-tuning ne change pas ce statut.